<a href="https://colab.research.google.com/github/kingdragonlord/ChatDev/blob/main/SEC_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================================================================
# CELL 1: SETUP & CONFIGURATION
# ===================================================================
import pandas as pd
import numpy as np
import os
import json
from tqdm.auto import tqdm
import os
import shutil
import sys

# --- 1. Mount Google Drive ---
print("--- Mounting Google Drive ---")
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully.")
else:
    print("Drive is already mounted.")

# --- Configuration ---
GDRIVE_PROJECT_BASE = "/content/drive/MyDrive/TFT_Project_Final/"
INTERMEDIATE_DIR = os.path.join(GDRIVE_PROJECT_BASE, "data_cache/sec_intermediate")
FINAL_OUTPUT_DIR = os.path.join(GDRIVE_PROJECT_BASE, "data_cache")
FINAL_FILENAME = "sec_engineered_features.parquet"

# Load the CIK map we created
cik_map_path = os.path.join(INTERMEDIATE_DIR, "symbol_cik_map.json")
with open(cik_map_path, 'r') as f:
    ticker_to_cik = json.load(f)

# The final list of symbols we will process
SYMBOLS_TO_PROCESS = list(ticker_to_cik.keys())

print(f"✅ Setup complete. Ready to engineer features for {len(SYMBOLS_TO_PROCESS)} symbols.")

--- Mounting Google Drive ---
Mounted at /content/drive
Drive mounted successfully.
✅ Setup complete. Ready to engineer features for 128 symbols.


In [ ]:
# ===================================================================
# CELL 2: ENGINEER FUNDAMENTAL MOMENTUM FEATURES (FINAL & CORRECT V4)
# ===================================================================
print("--- Engineering Fundamental Momentum Features from companyfacts ---")

def robust_pct_change(series, periods):
    old = series.shift(periods)
    return (series - old) / (old.abs() + 1e-9)

CONCEPTS_TO_TRACK = {
    'Revenues', 'NetIncomeLoss', 'EarningsPerShareBasic',
    'Assets', 'Liabilities', 'LongTermDebt'
}
all_fundamental_features = []

for symbol in tqdm(SYMBOLS_TO_PROCESS, desc="Processing Company Facts"):
    cik = ticker_to_cik.get(symbol)
    if not cik: continue
    file_path = os.path.join(INTERMEDIATE_DIR, "companyfacts", f"{cik}.json")
    if not os.path.exists(file_path): continue
    with open(file_path, 'r') as f: data = json.load(f)
    symbol_facts_dfs = []

    for concept in CONCEPTS_TO_TRACK:
        try:
            facts = data['facts']['us-gaap'][concept]['units']['USD']
            quarterly_facts = [f for f in facts if f['form'] in ['10-Q', '10-K'] and 'Q' in f['fp']]
            if not quarterly_facts: continue
            df = pd.DataFrame(quarterly_facts)
            df['end'] = pd.to_datetime(df['end'])
            df = df.sort_values('end').drop_duplicates(subset=['end'], keep='last')
            df[f'{concept}_qoq_growth'] = robust_pct_change(df['val'], periods=1)
            df[f'{concept}_yoy_growth'] = robust_pct_change(df['val'], periods=4)
            df[f'{concept}_yoy_acceleration'] = df[f'{concept}_yoy_growth'].diff(periods=1)
            df.set_index('end', inplace=True)
            daily_df = df.resample('D').ffill(limit=95)
            feature_cols = [col for col in daily_df.columns if 'growth' in col or 'acceleration' in col]
            if feature_cols:
                symbol_facts_dfs.append(daily_df[feature_cols])
        except KeyError:
            continue

    if symbol_facts_dfs:
        from functools import reduce
        final_symbol_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True, how='outer'), symbol_facts_dfs)
        final_symbol_df.reset_index(inplace=True)
        final_symbol_df.rename(columns={'end': 'date'}, inplace=True)
        final_symbol_df['symbol'] = symbol
        all_fundamental_features.append(final_symbol_df)

# --- Consolidate all symbols into one final dataframe ---
if all_fundamental_features:
    fundamental_features_df = pd.concat(all_fundamental_features, ignore_index=True)
    fundamental_features_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # --- THIS IS THE FINAL, CORRECT IMPUTATION LOGIC ---
    # Set multi-index to preserve symbol and date during fill
    fundamental_features_df = fundamental_features_df.set_index(['symbol', 'date'])
    # Group by the first level of the index (symbol) and fill
    fundamental_features_df = fundamental_features_df.groupby(level=0).ffill().bfill()
    # Return symbol and date back to columns
    fundamental_features_df.reset_index(inplace=True)
    # --- END OF FIX ---

    print("\n✅ Fundamental momentum feature engineering complete.")
    print(f"  -> Generated {len(fundamental_features_df):,} total data points.")

    numeric_cols = fundamental_features_df.select_dtypes(include=np.number).columns
    inf_check = fundamental_features_df[numeric_cols].isin([np.inf, -np.inf]).sum()


--- Engineering Fundamental Momentum Features from companyfacts ---


Processing Company Facts:   0%|          | 0/128 [00:00<?, ?it/s]


✅ Fundamental momentum feature engineering complete.
  -> Generated 716,400 total data points.


In [ ]:
# ===================================================================
# CELL 3: ENGINEER UNUSUAL FILING EVENT FEATURES
# ===================================================================
print("\n--- Engineering Unusual Filing Event Features from submissions ---")

all_filing_features = []

for symbol in tqdm(SYMBOLS_TO_PROCESS, desc="Processing Submissions History"):
    cik = ticker_to_cik.get(symbol)
    if not cik: continue

    file_path = os.path.join(INTERMEDIATE_DIR, "submissions", f"{cik}.parquet")
    if not os.path.exists(file_path): continue

    df = pd.read_parquet(file_path)
    df['filing_date'] = pd.to_datetime(df['filing_date'])
    df = df.sort_values('filing_date')

    # --- Feature Engineering ---
    # We only care about 8-K filings for this feature
    df_8k = df[df['form'] == '8-K'].copy()
    if df_8k.empty: continue

    # Count the number of 8-K filings in a rolling 30-day window
    df_8k.set_index('filing_date', inplace=True)
    # Create a series of 1s for each filing to be able to sum them up
    df_8k['count'] = 1

    # Resample to daily and sum the counts, then apply a rolling window
    daily_8k_counts = df_8k['count'].resample('D').sum()
    rolling_8k_counts = daily_8k_counts.rolling(window='30D').sum()

    final_df = rolling_8k_counts.reset_index()
    final_df.columns = ['date', '8k_filing_count_30d']
    final_df['symbol'] = symbol

    all_filing_features.append(final_df)

# --- Consolidate all symbols ---
if all_filing_features:
    filing_features_df = pd.concat(all_filing_features, ignore_index=True)
    print("\n✅ Filing event feature engineering complete.")
    print(f"  -> Generated {len(filing_features_df):,} total data points.")
    display(filing_features_df.head())
else:
    print("🛑 No filing event features were generated.")
    filing_features_df = pd.DataFrame()


--- Engineering Unusual Filing Event Features from submissions ---


Processing Submissions History:   0%|          | 0/128 [00:00<?, ?it/s]


✅ Filing event feature engineering complete.
  -> Generated 351,380 total data points.


,date,8k_filing_count_30d,symbol
0,2018-01-24,1.0,ABT
1,2018-01-25,1.0,ABT
2,2018-01-26,1.0,ABT
3,2018-01-27,1.0,ABT
4,2018-01-28,1.0,ABT


In [ ]:
# ===================================================================
# CELL 4: CONSOLIDATE AND SAVE ALL SEC FEATURES (FINAL & CORRECT V5)
# ===================================================================
print("\n--- Consolidating all new SEC features into a single file ---")

# --- Define paths needed for this cell ---
FINAL_OUTPUT_DIR = os.path.join(GDRIVE_PROJECT_BASE, "data_cache")
FINAL_FILENAME = "sec_engineered_features.parquet"
OUTPUT_DIR = os.path.join(GDRIVE_PROJECT_BASE, "data_cache/sec_intermediate")

# Start with the fundamental features from Cell 2 as the master list of companies.
if 'fundamental_features_df' not in locals() or fundamental_features_df.empty:
    print("🛑 Base fundamental features are missing. Cannot proceed.")
else:
    # This dataframe contains our ~128 target companies. This is our base.
    final_sec_features_df = fundamental_features_df.copy()

    # --- Robustly ensure all dataframes have UTC timezone before merging ---
    def ensure_utc(df, date_col='date'):
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col])
            if df[date_col].dt.tz is None:
                df[date_col] = df[date_col].dt.tz_localize('UTC')
            else:
                df[date_col] = df[date_col].dt.tz_convert('UTC')
        return df

    final_sec_features_df = ensure_utc(final_sec_features_df)

    # --- Merge the filing features using a LEFT join ---
    if 'filing_features_df' in locals() and not filing_features_df.empty:
        filing_features_df = ensure_utc(filing_features_df)
        final_sec_features_df = pd.merge(
            final_sec_features_df,
            filing_features_df,
            on=['date', 'symbol'],
            how='left' # CHANGED
        )

    # --- Merge Fails-to-Deliver data using a LEFT join ---
    fails_path = os.path.join(OUTPUT_DIR, "fails_to_deliver.parquet")
    if os.path.exists(fails_path):
        fails_df = pd.read_parquet(fails_path)
        fails_df = ensure_utc(fails_df, date_col='settlement_date')
        fails_df.rename(columns={'settlement_date': 'date'}, inplace=True)

        final_sec_features_df = pd.merge(
            final_sec_features_df,
            fails_df[['date', 'symbol', 'quantity']],
            on=['date', 'symbol'],
            how='left' # CHANGED
        )

    # --- FINAL, ROBUST IMPUTATION ---
    final_sec_features_df = final_sec_features_df.sort_values(['symbol', 'date'])

    # Fill count-based features with 0 (NaN means no events occurred)
    count_features = ['8k_filing_count_30d', 'quantity']
    for col in count_features:
        if col in final_sec_features_df.columns:
            final_sec_features_df[col] = final_sec_features_df[col].fillna(0)

    # Group by symbol and ffill/bfill everything else. This handles all other merge/calculation gaps.
    final_sec_features_df = final_sec_features_df.groupby('symbol', group_keys=False).apply(lambda group: group.ffill().bfill())

    # Final dropna as a safety net. This should drop very few, if any, rows.
    initial_rows = len(final_sec_features_df)
    final_sec_features_df.dropna(inplace=True)
    final_rows = len(final_sec_features_df)
    if initial_rows > final_rows:
        print(f"  - Note: dropna removed {initial_rows - final_rows} rows with persistent NaNs.")

    # --- Save the final consolidated file ---
    final_output_path = os.path.join(FINAL_OUTPUT_DIR, FINAL_FILENAME)
    final_sec_features_df.to_parquet(final_output_path, index=False)

    print(f"\n✅ All SEC features consolidated and saved successfully.")
    print(f"  -> Final file has {len(final_sec_features_df):,} rows for {final_sec_features_df['symbol'].nunique()} symbols.")
    print(f"  -> Saved to: {final_output_path}")
    print("\n--- Final DataFrame Info ---")
    final_sec_features_df.info(verbose=False, show_counts=True)


--- Consolidating all new SEC features into a single file ---


/tmp/ipython-input-1119673937.py:64: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_sec_features_df = final_sec_features_df.groupby('symbol', group_keys=False).apply(lambda group: group.ffill().bfill())


  - Note: dropna removed 2647 rows with persistent NaNs.

✅ All SEC features consolidated and saved successfully.
  -> Final file has 713,753 rows for 122 symbols.
  -> Saved to: /content/drive/MyDrive/TFT_Project_Final/data_cache/sec_engineered_features.parquet

--- Final DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 713753 entries, 338078 to 540687
Columns: 19 entries, symbol to quantity
dtypes: datetime64[ns, UTC](1), float64(17), object(1)
memory usage: 108.9+ MB


In [ ]:
# ===================================================================
# CELL 5: NAN FORENSIC ANALYSIS
# ===================================================================

print("\n--- Running NaN Forensic Analysis ---")

if 'final_sec_features_df' not in locals() or final_sec_features_df.empty:
    print("🛑 No data to analyze. Please run the consolidation cell first.")
else:
    df = final_sec_features_df.copy()

    # --- 1. Identify which symbols have NaN values ---
    nan_by_symbol = df.groupby('symbol').apply(lambda x: x.isnull().sum().sum())
    symbols_with_nans = nan_by_symbol[nan_by_symbol > 0].index.tolist()

    print(f"\nFound {len(symbols_with_nans)} symbols with at least one NaN value.")

    if symbols_with_nans:
        # --- 2. For each affected symbol, find WHICH columns have NaNs ---
        print("\n--- Detailed NaN Report per Symbol (Top 15) ---")

        for symbol in symbols_with_nans[:15]: # Show report for the first 15 affected symbols
            print(f"\n  --- Symbol: {symbol} ---")
            symbol_df = df[df['symbol'] == symbol]
            nan_counts = symbol_df.isnull().sum()
            nan_cols = nan_counts[nan_counts > 0]

            if not nan_cols.empty:
                for col, count in nan_cols.items():
                    percentage = (count / len(symbol_df)) * 100
                    print(f"    - Column '{col}': {count} NaNs ({percentage:.2f}%)")
            else:
                print("    - No NaNs found (this is unexpected, check logic).")

    # --- 3. The Final Imputation and Save (Moved from Cell 4) ---
    print("\n--- Applying Final Imputation and Saving File ---")

    # Group by symbol and then forward-fill AND back-fill all other features.
    imputed_df = df.groupby('symbol', group_keys=False).apply(lambda group: group.ffill().bfill())

    # Final check
    remaining_nans = imputed_df.isnull().sum().sum()
    if remaining_nans > 0:
        print(f"🛑 WARNING: {remaining_nans} NaNs remain after imputation. Dropping affected rows.")
        imputed_df.dropna(inplace=True)
    else:
        print("✅ Imputation successful. No remaining NaNs.")

    final_output_path = os.path.join(FINAL_OUTPUT_DIR, FINAL_FILENAME)
    imputed_df.to_parquet(final_output_path, index=False)

    print(f"\n✅ Final file saved successfully.")
    print(f"  -> Final file has {len(imputed_df):,} rows for {imputed_df['symbol'].nunique()} symbols.")
    print(f"  -> Saved to: {final_output_path}")


--- Running NaN Forensic Analysis ---


/tmp/ipython-input-3156801361.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nan_by_symbol = df.groupby('symbol').apply(lambda x: x.isnull().sum().sum())



Found 0 symbols with at least one NaN value.

--- Applying Final Imputation and Saving File ---


/tmp/ipython-input-3156801361.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  imputed_df = df.groupby('symbol', group_keys=False).apply(lambda group: group.ffill().bfill())


✅ Imputation successful. No remaining NaNs.

✅ Final file saved successfully.
  -> Final file has 713,753 rows for 122 symbols.
  -> Saved to: /content/drive/MyDrive/TFT_Project_Final/data_cache/sec_engineered_features.parquet


In [ ]:
# ===================================================================
# CELL 5: MASTER VERIFICATION OF FINAL SEC FEATURE FILE
# ===================================================================
import pandas as pd
import numpy as np
import os

print("--- Running Master Verification on sec_engineered_features.parquet ---")

# --- Configuration ---
FINAL_FILE_PATH = os.path.join(FINAL_OUTPUT_DIR, FINAL_FILENAME)
# The expected count is the number of symbols we found a CIK for
EXPECTED_SYMBOLS_COUNT = len(SYMBOLS_TO_PROCESS)

# --- 1. File Existence & Loading ---
if not os.path.exists(FINAL_FILE_PATH):
    print(f"🛑 FAILED: Final feature file not found at {FINAL_FILE_PATH}")
else:
    print(f"✅ Found final file: {FINAL_FILE_PATH}")
    try:
        df = pd.read_parquet(FINAL_FILE_PATH)
        print(f"  - Successfully loaded file with {len(df):,} rows.")

        # --- 2. Content & Data Integrity Verification ---
        print("\n--- Verifying Content and Data Integrity ---")

        # a) Symbol Completeness
        found_symbols = df['symbol'].nunique()
        if found_symbols == EXPECTED_SYMBOLS_COUNT:
            print(f"  - ✅ Symbols: PASSED. Found data for all {found_symbols} expected symbols.")
        else:
            print(f"  - 🛑 Symbols: WARNING. Found {found_symbols}, but expected {EXPECTED_SYMBOLS_COUNT}.")

        # b) Date Range
        min_date = df['date'].min().date()
        max_date = df['date'].max().date()
        print(f"  - ✅ Date Range: PASSED. Data spans from {min_date} to {max_date}.")

        # c) NaN Value Check
        nan_sum = df.isnull().sum().sum()
        if nan_sum == 0:
            print(f"  - ✅ NaN Check: PASSED. No missing values found.")
        else:
            print(f"  - 🛑 NaN Check: FAILED. Found {nan_sum} total missing values. Rerun required.")

        # d) Infinity Value Check
        numeric_cols = df.select_dtypes(include=np.number).columns
        inf_sum = np.isinf(df[numeric_cols]).sum().sum()
        if inf_sum == 0:
            print(f"  - ✅ Infinity Check: PASSED. No infinite values found.")
        else:
            print(f"  - 🛑 Infinity Check: FAILED. Found {inf_sum} infinite values. Rerun required.")

    except Exception as e:
        print(f"🛑 FAILED: Could not load or inspect the parquet file. Error: {e}")

print("\n--- Master Verification Complete ---")

--- Running Master Verification on sec_engineered_features.parquet ---
✅ Found final file: /content/drive/MyDrive/TFT_Project_Final/data_cache/sec_engineered_features.parquet
  - Successfully loaded file with 713,753 rows.

--- Verifying Content and Data Integrity ---
  - 🛑 Symbols: WARNING. Found 122, but expected 128.
  - ✅ Date Range: PASSED. Data spans from 2008-03-28 to 2025-08-31.
  - ✅ NaN Check: PASSED. No missing values found.
  - ✅ Infinity Check: PASSED. No infinite values found.

--- Master Verification Complete ---


In [ ]:
# ===================================================================
# CELL 8: FINAL SYMBOL RECONCILIATION
# ===================================================================

print("--- Running Final Symbol Reconciliation ---")

# --- 1. Load the CIK map to get our initial list of companies ---
cik_map_path = os.path.join(OUTPUT_DIR, "symbol_cik_map.json")
with open(cik_map_path, 'r') as f:
    ticker_to_cik = json.load(f)

# This is the list of all companies we TRIED to get fundamental data for
expected_symbols = set(ticker_to_cik.keys())

# --- 2. Load the final feature file to see who survived ---
final_file_path = os.path.join(FINAL_OUTPUT_DIR, FINAL_FILENAME)
df_final = pd.read_parquet(final_file_path)
actual_symbols = set(df_final['symbol'].unique())

# --- 3. Find the symbols that were dropped ---
dropped_symbols = expected_symbols - actual_symbols

print(f"\nInitial company universe size: {len(expected_symbols)}")
print(f"Final universe size after processing: {len(actual_symbols)}")
print(f"Total symbols dropped: {len(dropped_symbols)}")

if dropped_symbols:
    print("\n--- List of Dropped Symbols ---")
    # We know SPY, QQQ, DIA were expected to be dropped. Let's show the others.
    known_etfs = {"SPY", "QQQ", "DIA"}
    unexpected_drops = sorted(list(dropped_symbols - known_etfs))

    print("  - Known ETFs (correctly dropped):")
    for symbol in sorted(list(known_etfs.intersection(dropped_symbols))):
        print(f"    - {symbol}")

    if unexpected_drops:
        print("\n  - Unexpectedly Dropped Companies (due to sparse data):")
        for symbol in unexpected_drops:
            print(f"    - {symbol}")

    print("\n--- FINAL RECOMMENDATION ---")
    print("To create the final, stable modeling universe, remove all of the above")
    print("dropped symbols from the `SYMBOLS_TO_FETCH` list in your main pipeline scripts.")

    print("\n--- Final, Clean Symbol List for v4.3 Pipeline ---")
    final_clean_list = sorted(list(actual_symbols))
    print(final_clean_list)

else:
    print("\n✅ SUCCESS: All expected companies were present in the final file.")

--- Running Final Symbol Reconciliation ---

Initial company universe size: 128
Final universe size after processing: 122
Total symbols dropped: 6

--- List of Dropped Symbols ---
  - Known ETFs (correctly dropped):
    - DIA
    - QQQ
    - SPY

  - Unexpectedly Dropped Companies (due to sparse data):
    - DIS
    - GSK
    - SAP

--- FINAL RECOMMENDATION ---
To create the final, stable modeling universe, remove all of the above
dropped symbols from the `SYMBOLS_TO_FETCH` list in your main pipeline scripts.

--- Final, Clean Symbol List for v4.3 Pipeline ---
['AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'ADI', 'AEP', 'AIG', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'AVGO', 'AXP', 'BA', 'BAC', 'BIIB', 'BKNG', 'BLK', 'BMY', 'BSX', 'C', 'CAT', 'CI', 'CMCSA', 'CMG', 'CMI', 'COF', 'COP', 'COST', 'CRM', 'CSCO', 'CSX', 'CVS', 'CVX', 'DE', 'DHR', 'DOW', 'DUK', 'DVN', 'EOG', 'ETN', 'EXC', 'F', 'FCX', 'FDX', 'GD', 'GILD', 'GLD', 'GM', 'GOOGL', 'GS', 'HAL', 'HD', 'HPE', 'HPQ', 'IBM', 'INTC', 'INTU', 'ISRG', 'JNJ